In [ ]:
library("tidyverse")
options(repr.plot.width=5, repr.plot.height=5)#Default=7x7

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")


for (package in c("DEqMS","WGCNA")) {
    BiocManager::install(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}
#CRAN
for (package in c("RColorBrewer", "ggrepel")) {
    install.packages(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}

In [ ]:
library(limma)


group_df <- read.csv(
  "./Data/CA19_9_difference_groups.csv",
  stringsAsFactors = FALSE
)
head(group_df)
nrow(group_df)

In [ ]:
fileDir <- "./Data/"
fileName <- "pro_data_common_dis.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data_dis <- temp
dim(temp)

fileDir <- "./Data/"
fileName <- "pro_sample_dis.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path)
colnames(temp)[1] <- "SampleID"
head(temp)
pro_sample_dis <- temp


fileDir <- "./Data/"
fileName <- "pro_data_common_val_Assay.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data_val <- temp
dim(temp)

fileDir <- "./Data/"
fileName <- "pro_sample_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path)
colnames(temp)[1] <- "SampleID"
head(temp)
pro_sample_val <- temp


In [ ]:
temp <- pro_sample_val

temp <- temp %>%
  mutate(
    progress_group = case_when(
      progress == 0 ~ "0",
      progress %in% c(400, 410) ~ "1",
      progress %in% c(420, 430, 440) ~ "2",
      TRUE ~ NA_character_
    ),
    progress_group = factor(
      progress_group,
      levels = c("0", "1", "2")
    )
  )

table(temp$progress_group, useNA = "ifany")
class(temp$progress_group)
pro_sample_val <- temp

In [ ]:
# library(limma)


# group_df <- read.csv(
#   "./Data/CA19_9_difference_groups.csv",
#   stringsAsFactors = FALSE
# )

# expr <- as.matrix(pro_data)
# head(expr)
# rownames(expr) <- trimws(as.character(rownames(expr)))
# group_df$SampleID <- trimws(as.character(group_df$SampleID))
# group_df2 <- group_df[
#   match(rownames(expr)[rownames(expr) %in% group_df$SampleID],
#         group_df$SampleID),
#   ,
#   drop = FALSE
# ]
# head(group_df2)
# group_df <- group_df2

In [ ]:
# GroupがLowerのSampleIDを抽出
lower_ids <- group_df$SampleID[group_df$Group == "Lower"]

# pro_dataから抽出
pro_data_upper <- pro_data_val[rownames(pro_data_val) %in% lower_ids, ]

# pro_sampleから抽出（SampleID列がある場合）
pro_sample_upper <- pro_sample_val[pro_sample_val$SampleID %in% lower_ids, ]

nrow(pro_data_upper)
nrow(pro_sample_upper)
head(pro_data_upper)
head(pro_sample_upper)

In [ ]:
# GroupがLowerのSampleIDを抽出
lower_ids <- group_df$SampleID[group_df$Group == "Upper"]

# pro_dataから抽出
pro_data_upper <- pro_data_val[rownames(pro_data_val) %in% lower_ids, ]

# pro_sampleから抽出（SampleID列がある場合）
pro_sample_upper <- pro_sample_val[pro_sample_val$SampleID %in% lower_ids, ]

nrow(pro_data_upper)
nrow(pro_sample_upper)
head(pro_data_upper)
head(pro_sample_upper)

# Design Matrix(Lower by progress)

In [ ]:
sample_group <- c()
sample_gender <- c()
sample_age <- c()
sample_bmi <- c()

# pro_data_upperの行順に合わせてサンプル情報を取得
idx <- match(rownames(pro_data_lower), pro_sample_lower$SampleID)

sample_group <- pro_sample_lower$progress_group[idx]
sample_gender <- pro_sample_lower$sex[idx]
sample_age <- pro_sample_lower$fvage[idx]
sample_bmi <- pro_sample_lower$bmi[idx]

# 変換前の値を確認
print(table(sample_group, useNA = "ifany"))
group_order <- c(0, 1, 2)
sample_group <- factor(sample_group, levels = group_order, labels = c("Control", "Early_Stage", "Advanced_Stage"))
group_order <- c(1,2)
sample_gender <- factor(sample_gender, levels = group_order, labels = c("Male", "Female"))

design <- model.matrix(~0 + sample_group + sample_gender + sample_age + sample_bmi)
colnames(design) <- c("Control", "Early_Stage", "Advanced_Stage", "Female", "Age", "BMI")
design


## Check

In [ ]:
#check
tempD <- apply(design, 2, sum)
tempD <- tibble(Group = names(tempD), DesignMatrix = tempD)
tempD2 <- pro_sample_lower
# print(tempD)
# str(tempD2$progress_group)

tempD2 %>%
    dplyr::mutate(Group = factor(progress_group, levels = c(0, 1, 2), labels = c("Control", "Early_Stage", "Advanced_Stage"))) %>%
    # dplyr::mutate(Group = as.character(progress_group)) %>%
    dplyr::group_by(Group) %>%
    dplyr::summarize(nSamples = n()) %>%
    dplyr::left_join(tempD, by = "Group")

## Result of Limma

In [ ]:
dim(t(pro_data_lower))
dim(design)

In [ ]:
#Limmaでやってみる
library(limma)
pro_data_lower_t <- t(pro_data_lower)


fit <- lmFit(pro_data_lower_t, design)
contrast <- makeContrasts(
    Early_vs_Control = Early_Stage - Control,
    Advanced_vs_Control = Advanced_Stage - Control,
    Advanced_vs_Early = Advanced_Stage - Early_Stage,
    levels = design
)
fit2 <- contrasts.fit(fit, contrast)
fit2 <- eBayes(fit2)

#Early_vs_Control
print("Early_vs_Control")
result_Early_vs_Control <- topTable(
    fit2,
    coef = "Early_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Early_vs_Control)

sig_raw <- sum(result_Early_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Early_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Early_vs_Control$adj.P.Val < 0.05 &
    abs(result_Early_vs_Control$logFC) > 0.5
)


#Advanced_vs_Control
print("Advanced_vs_Control")
result_Advanced_vs_Control <- topTable(
    fit2,
    coef = "Advanced_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Control)

sig_raw <- sum(result_Advanced_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Control$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Control$logFC) > 0.5
)

#Advanced_vs_Early
print("Advanced_vs_Early")
result_Advanced_vs_Early <- topTable(
    fit2,
    coef = "Advanced_vs_Early",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Early)

sig_raw <- sum(result_Advanced_vs_Early$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Early$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Early$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Early$logFC) > 0.5
)

# Export result data

In [ ]:
contrast_names <- colnames(contrast)

DEPlist <- lapply(contrast_names, function(coef_name) {
  
  result <- topTable(
    fit2,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH"
  )
  
  result %>%
    mutate(
      DEP = case_when(
        adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
        adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
        TRUE ~ "NS"
      )
    ) %>%
    filter(DEP != "NS") %>%
    arrange(adj.P.Val)
})

names(DEPlist) <- contrast_names
head(DEPlist)

DEP_combined <- bind_rows(
  lapply(names(DEPlist), function(name) {
    DEPlist[[name]] %>%
      mutate(Protein = rownames(DEPlist[[name]]),　Contrast = name)%>%
      relocate(Protein)
  })
)


In [ ]:
library(ggrepel)

contrast_names <- colnames(contrast)

volcano_plots <- lapply(contrast_names, function(coef_name) {
  
  # そのcontrastの結果を取得
  result_table <- topTable(
    fit2,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH"
  )
  
  # DEP分類
  plot_data <- result_table %>%
    mutate(
      threshold = case_when(
        adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
        adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
        TRUE ~ "NS"
      )
    )
  
  # ラベルするタンパク質
  label_data <- bind_rows(
    plot_data %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(adj.P.Val) %>%
      head(10),
    
    plot_data %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(desc(abs(logFC))) %>%
      head(10)
  ) %>%
    distinct()
  
  # Volcano plot
  p <- ggplot(
    plot_data,
    aes(x = logFC, y = -log10(P.Value), color = threshold)
  ) +
    geom_point(alpha = 1, size = 1.5) +
    
    scale_color_manual(
      values = c(
        "Up" = "#E41A1C",
        "Down" = "#377EB8",
        "NS" = "grey"
      )
    ) +
    
    geom_text_repel(
      data = label_data,
      aes(label = rownames(label_data)),
      size = 5,
      box.padding = 0.5,
      max.overlaps = Inf
    ) +
    
    geom_vline(
      xintercept = c(-0.5, 0.5),
      linetype = "dashed",
      color = "black",
      alpha = 0.5
    ) +
    
    geom_hline(
      yintercept = -log10(0.05),
      linetype = "dashed",
      color = "black",
      alpha = 0.5
    ) +
    
    theme_minimal() +
    
    labs(
      title = paste0("Volcano Plot: ", coef_name),
      subtitle = "Adjusted for Sex, Age, and BMI",
      x = "log2 Fold Change",
      y = "-log10 P-value"
    ) +
    
    theme(legend.position = "none")
  
  return(p)
})

# 名前を付ける
names(volcano_plots) <- contrast_names

volcano_plots$Early_vs_Control
volcano_plots$Advanced_vs_Control
volcano_plots$Advanced_vs_Early

# Design Matrix(Upper by progress)

In [ ]:
sample_group <- c()
sample_gender <- c()
sample_age <- c()
sample_bmi <- c()

# pro_data_upperの行順に合わせてサンプル情報を取得
idx <- match(rownames(pro_data_upper), pro_sample_upper$SampleID)

sample_group <- pro_sample_upper$progress_group[idx]
sample_gender <- pro_sample_upper$sex[idx]
sample_age <- pro_sample_upper$fvage[idx]
sample_bmi <- pro_sample_upper$bmi[idx]

# 変換前の値を確認
print(table(sample_group, useNA = "ifany"))
group_order <- c(0, 1, 2)
sample_group <- factor(sample_group, levels = group_order, labels = c("Control", "Early_Stage", "Advanced_Stage"))
group_order <- c(1,2)
sample_gender <- factor(sample_gender, levels = group_order, labels = c("Male", "Female"))

design <- model.matrix(~0 + sample_group + sample_gender + sample_age + sample_bmi)
colnames(design) <- c("Control", "Early_Stage", "Advanced_Stage", "Female", "Age", "BMI")
design


## Check

In [ ]:
#check
tempD <- apply(design, 2, sum)
tempD <- tibble(Group = names(tempD), DesignMatrix = tempD)
tempD2 <- pro_sample_upper
# print(tempD)
# str(tempD2$progress_group)

tempD2 %>%
    dplyr::mutate(Group = factor(progress_group, levels = c(0, 1, 2), labels = c("Control", "Early_Stage", "Advanced_Stage"))) %>%
    # dplyr::mutate(Group = as.character(progress_group)) %>%
    dplyr::group_by(Group) %>%
    dplyr::summarize(nSamples = n()) %>%
    dplyr::left_join(tempD, by = "Group")

## Result of Limma

In [ ]:
dim(t(pro_data_upper))
dim(design)

In [ ]:
#Limmaでやってみる
library(limma)
pro_data_upper_t <- t(pro_data_upper)

fit <- lmFit(pro_data_upper_t, design)
contrast <- makeContrasts(
    Early_vs_Control = Early_Stage - Control,
    Advanced_vs_Control = Advanced_Stage - Control,
    Advanced_vs_Early = Advanced_Stage - Early_Stage,
    levels = design
)
fit2 <- contrasts.fit(fit, contrast)
fit2 <- eBayes(fit2)

#Early_vs_Control
print("Early_vs_Control")
result_Early_vs_Control <- topTable(
    fit2,
    coef = "Early_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Early_vs_Control)

sig_raw <- sum(result_Early_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Early_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Early_vs_Control$adj.P.Val < 0.05 &
    abs(result_Early_vs_Control$logFC) > 0.5
)


#Advanced_vs_Control
print("Advanced_vs_Control")
result_Advanced_vs_Control <- topTable(
    fit2,
    coef = "Advanced_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Control)

sig_raw <- sum(result_Advanced_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Control$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Control$logFC) > 0.5
)

#Advanced_vs_Early
print("Advanced_vs_Early")
result_Advanced_vs_Early <- topTable(
    fit2,
    coef = "Advanced_vs_Early",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Early)

sig_raw <- sum(result_Advanced_vs_Early$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Early$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Early$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Early$logFC) > 0.5
)

# Export result data

In [ ]:
contrast_names <- colnames(contrast)

DEPlist <- lapply(contrast_names, function(coef_name) {
  
  result <- topTable(
    fit2,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH"
  )
  
  result %>%
    mutate(
      DEP = case_when(
        adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
        adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
        TRUE ~ "NS"
      )
    ) %>%
    filter(DEP != "NS") %>%
    arrange(adj.P.Val)
})

names(DEPlist) <- contrast_names
head(DEPlist)

DEP_combined <- bind_rows(
  lapply(names(DEPlist), function(name) {
    DEPlist[[name]] %>%
      mutate(Protein = rownames(DEPlist[[name]]),　Contrast = name)%>%
      relocate(Protein)
  })
)


In [ ]:
library(ggrepel)

contrast_names <- colnames(contrast)

volcano_plots <- lapply(contrast_names, function(coef_name) {
  
  # そのcontrastの結果を取得
  result_table <- topTable(
    fit2,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH"
  )
  
  # DEP分類
  plot_data <- result_table %>%
    mutate(
      threshold = case_when(
        adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
        adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
        TRUE ~ "NS"
      )
    )
  
  # ラベルするタンパク質
  label_data <- bind_rows(
    plot_data %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(adj.P.Val) %>%
      head(10),
    
    plot_data %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(desc(abs(logFC))) %>%
      head(10)
  ) %>%
    distinct()
  
  # Volcano plot
  p <- ggplot(
    plot_data,
    aes(x = logFC, y = -log10(P.Value), color = threshold)
  ) +
    geom_point(alpha = 1, size = 1.5) +
    
    scale_color_manual(
      values = c(
        "Up" = "#E41A1C",
        "Down" = "#377EB8",
        "NS" = "grey"
      )
    ) +
    
    geom_text_repel(
      data = label_data,
      aes(label = rownames(label_data)),
      size = 5,
      box.padding = 0.5,
      max.overlaps = Inf
    ) +
    
    geom_vline(
      xintercept = c(-0.5, 0.5),
      linetype = "dashed",
      color = "black",
      alpha = 0.5
    ) +
    
    geom_hline(
      yintercept = -log10(0.05),
      linetype = "dashed",
      color = "black",
      alpha = 0.5
    ) +
    
    theme_minimal() +
    
    labs(
      title = paste0("Volcano Plot: ", coef_name),
      subtitle = "Adjusted for Sex, Age, and BMI",
      x = "log2 Fold Change",
      y = "-log10 P-value"
    ) +
    
    theme(legend.position = "none")
  
  return(p)
})

# 名前を付ける
names(volcano_plots) <- contrast_names

volcano_plots$Early_vs_Control
volcano_plots$Advanced_vs_Control
volcano_plots$Advanced_vs_Early

# Design Matrix(lower by status)

In [ ]:
sample_group <- c()
sample_gender <- c()
sample_age <- c()
sample_bmi <- c()

for (sample in rownames(pro_data_lower)) {
    # group <- pro_sample_lower[rownames(pro_data_lower) %in% pro_sample_lower$SampleID,]$CaCo_Status
    matched_row <- pro_sample_lower[pro_sample_lower$SampleID == sample, ]
    
    group <- if(nrow(matched_row) > 0) matched_row$CaCo_Status else NA
    sample_group <- c(sample_group, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$sex else NA
    sample_gender <- c(sample_gender, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$fvage else NA
    sample_age <- c(sample_age, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$bmi else NA
    sample_bmi <- c(sample_bmi, group)
    
    # print(sample_group)
    # table(rownames(pro_data_lower) %in% pro_sample_lower$SampleID)
}
group_order <- c(0, 1)
sample_group <- factor(sample_group, levels = group_order, labels = c("Non.affected", "Affected"))
group_order <- c(1,2)
sample_gender <- factor(sample_gender, levels = group_order, labels = c("Male", "Female"))

design <- model.matrix(~0 + sample_group + sample_gender + sample_age + sample_bmi)
colnames(design) <- c("Non.affected", "Affected", "Female", "Age", "BMI")
design

## Check

In [ ]:
#check
tempD <- apply(design, 2, sum)
tempD <- tibble(Group = names(tempD), DesignMatrix = tempD)
tempD2 <- pro_sample_lower
# print(tempD)
# str(tempD2$CaCo_Status)

tempD2 %>%
    dplyr::mutate(Group = factor(CaCo_Status, levels = c(0, 1), labels = c("Non.affected", "Affected"))) %>%
    # dplyr::mutate(Group = as.character(CaCo_Status)) %>%
    dplyr::group_by(Group) %>%
    dplyr::summarize(nSamples = n()) %>%
    dplyr::left_join(tempD, by = "Group")

## Result of Limma

In [ ]:
dim(t(pro_data_lower))
dim(design)

In [ ]:
#Limmaでやってみる
library(limma)
pro_data_lower_t <- t(pro_data_lower)


fit <- lmFit(pro_data_lower_t, design)
contrast <- makeContrasts(
    Affected_vs_Non_Affected = Affected - Non.affected,
    levels = design
)
fit2 <- contrasts.fit(fit, contrast)
fit2 <- eBayes(fit2)
result_table <- topTable(fit2,coef = "Affected_vs_Non_Affected", number = Inf, adjust.method = "BH")

# rownames(result_table) <- result_table[[1]]

# result_table <- result_table[, -1]
head(result_table)
sig_raw <- sum(result_table$P.Value < 0.05)
sig_adj <- sum(result_table$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")
sum(result_table$adj.P.Val < 0.05 & abs(result_table$logFC) > 0.5)

# Export result data

In [ ]:
temp <- result_table %>%
  mutate(
    DEP = case_when(
      adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
      adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
      TRUE ~ "NS"
    )
  )
head(temp)
DEPlist <- temp %>%
    filter(DEP != "NS") %>%
    arrange(adj.P.Val)


dim(DEPlist)
DEPlist

#Export DEP list


In [ ]:
library(ggrepel)
plot_data <- result_table %>%
  mutate(
    threshold = case_when(
      adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
      adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
      TRUE ~ "NS"
    )
  )

label_data <- bind_rows(
    data = plot_data %>% filter(adj.P.Val < 0.05) %>% arrange(adj.P.Val) %>% head(10),
    data = plot_data %>% filter(adj.P.Val < 0.05) %>% arrange(desc(abs(logFC))) %>% head(10)
) %>% distinct()

p <- ggplot(plot_data, aes(x = logFC, y = -log10(P.Value), color = threshold)) +
  geom_point(alpha = 1, size = 1.5) +
  scale_color_manual(values = c("Up" = "#E41A1C", "Down" = "#377EB8", "NS" = "grey")) +
  
  geom_text_repel(
    data = label_data,
    aes(label = rownames(label_data)),
    size = 5,
    box.padding = 0.5,
    max.overlaps = Inf
  ) +
  
  geom_vline(xintercept = c(-0.5, 0.5), linetype = "dashed", color = "black", alpha = 0.5) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black", alpha = 0.5) +
  
  theme_minimal() +
  labs(
    title = "Volcano Plot: Affected vs Non-affected",
    subtitle = "Adjusted for Sex, Age, and BMI",
    x = "log2 Fold Change",
    y = "-log10 P-value"
  ) +
  theme(legend.position = "none")

print(p)

# Design Matrix(upper by status)

In [ ]:
sample_group <- c()
sample_gender <- c()
sample_age <- c()
sample_bmi <- c()

for (sample in rownames(pro_data_upper)) {
    # group <- pro_sample_upper[rownames(pro_data_upper) %in% pro_sample_upper$SampleID,]$CaCo_Status
    matched_row <- pro_sample_upper[pro_sample_upper$SampleID == sample, ]
    
    group <- if(nrow(matched_row) > 0) matched_row$CaCo_Status else NA
    sample_group <- c(sample_group, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$sex else NA
    sample_gender <- c(sample_gender, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$fvage else NA
    sample_age <- c(sample_age, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$bmi else NA
    sample_bmi <- c(sample_bmi, group)
    
    # print(sample_group)
    # table(rownames(pro_data_upper) %in% pro_sample_upper$SampleID)
}
group_order <- c(0, 1)
sample_group <- factor(sample_group, levels = group_order, labels = c("Non.affected", "Affected"))
group_order <- c(1,2)
sample_gender <- factor(sample_gender, levels = group_order, labels = c("Male", "Female"))

design <- model.matrix(~0 + sample_group + sample_gender + sample_age + sample_bmi)
colnames(design) <- c("Non.affected", "Affected", "Female", "Age", "BMI")
design

## Check

In [ ]:
#check
tempD <- apply(design, 2, sum)
tempD <- tibble(Group = names(tempD), DesignMatrix = tempD)
tempD2 <- pro_sample_upper
# print(tempD)
# str(tempD2$CaCo_Status)

tempD2 %>%
    dplyr::mutate(Group = factor(CaCo_Status, levels = c(0, 1), labels = c("Non.affected", "Affected"))) %>%
    # dplyr::mutate(Group = as.character(CaCo_Status)) %>%
    dplyr::group_by(Group) %>%
    dplyr::summarize(nSamples = n()) %>%
    dplyr::left_join(tempD, by = "Group")

## Result of Limma

In [ ]:
dim(t(pro_data_upper))
dim(design)

In [ ]:
#Limmaでやってみる
library(limma)
pro_data_upper_t <- t(pro_data_upper)


fit <- lmFit(pro_data_upper_t, design)
contrast <- makeContrasts(
    Affected_vs_Non_Affected = Affected - Non.affected,
    levels = design
)
fit2 <- contrasts.fit(fit, contrast)
fit2 <- eBayes(fit2)
result_table <- topTable(fit2,coef = "Affected_vs_Non_Affected", number = Inf, adjust.method = "BH")

# rownames(result_table) <- result_table[[1]]

# result_table <- result_table[, -1]
head(result_table)
sig_raw <- sum(result_table$P.Value < 0.05)
sig_adj <- sum(result_table$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")
sum(result_table$adj.P.Val < 0.05 & abs(result_table$logFC) > 0.5)

# Export result data

In [ ]:
temp <- result_table %>%
  mutate(
    DEP = case_when(
      adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
      adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
      TRUE ~ "NS"
    )
  )
head(temp)
DEPlist <- temp %>%
    filter(DEP != "NS") %>%
    arrange(adj.P.Val)


dim(DEPlist)
DEPlist

#Export DEP list


In [ ]:
library(ggrepel)
plot_data <- result_table %>%
  mutate(
    threshold = case_when(
      adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
      adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
      TRUE ~ "NS"
    )
  )

label_data <- bind_rows(
    data = plot_data %>% filter(adj.P.Val < 0.05) %>% arrange(adj.P.Val) %>% head(10),
    data = plot_data %>% filter(adj.P.Val < 0.05) %>% arrange(desc(abs(logFC))) %>% head(10)
) %>% distinct()

p <- ggplot(plot_data, aes(x = logFC, y = -log10(P.Value), color = threshold)) +
  geom_point(alpha = 1, size = 1.5) +
  scale_color_manual(values = c("Up" = "#E41A1C", "Down" = "#377EB8", "NS" = "grey")) +
  
  geom_text_repel(
    data = label_data,
    aes(label = rownames(label_data)),
    size = 5,
    box.padding = 0.5,
    max.overlaps = Inf
  ) +
  
  geom_vline(xintercept = c(-0.5, 0.5), linetype = "dashed", color = "black", alpha = 0.5) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black", alpha = 0.5) +
  
  theme_minimal() +
  labs(
    title = "Volcano Plot: Affected vs Non-affected",
    subtitle = "Adjusted for Sex, Age, and BMI",
    x = "log2 Fold Change",
    y = "-log10 P-value"
  ) +
  theme(legend.position = "none")

print(p)